In [ ]:
#=======================bài1=======================
import copy
from heapq import heappush, heappop

n = 4  # Đây là bài toán 15-puzzle, vì vậy n = 4

# Các hướng di chuyển: dưới, trái, trên, phải
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class PriorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, key):
        heappush(self.heap, key)

    def pop(self):
        return heappop(self.heap)

    def empty(self):
        return len(self.heap) == 0


class Node:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return (self.costs + self.levels) < (nxt.costs + nxt.levels)


def manhattan_distance(mats, final):
    """Tính tổng khoảng cách Manhattan của các ô trong ma trận so với vị trí của chúng trong ma trận mục tiêu."""
    distance = 0
    for i in range(n):
        for j in range(n):
            tile = mats[i][j]
            if tile != 0:
                final_pos = find_position(final, tile)
                distance += abs(i - final_pos[0]) + abs(j - final_pos[1])
    return distance

def find_position(matrix, value):
    """Tìm vị trí của giá trị trong ma trận mục tiêu."""
    for i in range(n):
        for j in range(n):
            if matrix[i][j] == value:
                return (i, j)

def new_node(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final):
    new_mats = copy.deepcopy(mats)
    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]
    costs = manhattan_distance(new_mats, final)
    return Node(parent, new_mats, new_empty_tile_posi, costs, levels)


def print_matrix(mats):
    for i in range(n):
        for j in range(n):
            print(f"{mats[i][j]:2d}", end=" ")
        print()
    print()


def is_safe(x, y):
    return 0 <= x < n and 0 <= y < n


def print_path(root):
    if root is None:
        return
    print_path(root.parent)
    print_matrix(root.mats)


def solve(initial, empty_tile_posi, final):
    pq = PriorityQueue()
    root = Node(None, initial, empty_tile_posi, manhattan_distance(initial, final), 0)
    pq.push(root)
    visited = set()

    while not pq.empty():
        minimum = pq.pop()
        state_tuple = tuple(tuple(row) for row in minimum.mats)
        if state_tuple in visited:
            continue
        visited.add(state_tuple)

        if minimum.costs == 0:
            print_path(minimum)
            return

        for i in range(4):
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]

            if is_safe(new_tile_posi[0], new_tile_posi[1]):
                child = new_node(minimum.mats, minimum.empty_tile_posi, new_tile_posi, minimum.levels + 1, minimum, final)
                pq.push(child)

# Ví dụ về ma trận ban đầu và mục tiêu cho bài toán 15-puzzle
initial = [
    [ 1, 2, 3, 4 ],
    [ 5, 6, 7, 8 ],
    [ 9, 10, 11, 0 ],
    [ 13, 14, 15, 12 ]
]

final = [
    [ 1, 2, 3, 4 ],
    [ 5, 6, 7, 8 ],
    [ 9, 10, 11, 12 ],
    [ 13, 14, 15, 0 ]
]

empty_tile_posi = [2, 3]
solve(initial, empty_tile_posi, final)


#=======================Bài 2===============================
class Graph:
    def __init__(self, adjac_lis, heuristic):
        self.adjac_lis = adjac_lis   # danh sách kề
        self.heuristic = heuristic   # hàm h(n)

    def get_neighbors(self, v):
        return self.adjac_lis.get(v, [])

    def h(self, n):
        return self.heuristic.get(n, 0)

    def a_star_algorithm(self, start, stop):
        open_set = set([start])
        closed_set = set()

        g = {}  # chi phí từ start → node
        g[start] = 0

        parent = {}
        parent[start] = start

        while len(open_set) > 0:
            n = None

            # chọn node có f(n) nhỏ nhất
            for v in open_set:
                if n is None or g[v] + self.h(v) < g[n] + self.h(n):
                    n = v

            if n is None:
                print("Không tồn tại đường đi")
                return None

            # nếu tới đích
            if n == stop:
                path = []
                while parent[n] != n:
                    path.append(n)
                    n = parent[n]
                path.append(start)
                path.reverse()

                print("Đường đi:", path)
                print("Chi phí:", g[stop])
                return path

            # duyệt hàng xóm
            for (m, weight) in self.get_neighbors(n):
                if m not in open_set and m not in closed_set:
                    open_set.add(m)
                    parent[m] = n
                    g[m] = g[n] + weight
                else:
                    # nếu tìm được đường tốt hơn
                    if g[m] > g[n] + weight:
                        g[m] = g[n] + weight
                        parent[m] = n

                        if m in closed_set:
                            closed_set.remove(m)
                            open_set.add(m)

            open_set.remove(n)
            closed_set.add(n)

        print("Không tìm thấy đường đi")
        return None

adjac_lis = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5), ('E', 2)],
    'C': [('D', 2), ('E', 4)],
    'D': [('E', 1)],
    'E': []
}

# heuristic (ước lượng khoảng cách đến đích E)
heuristic = {
    'A': 5,
    'B': 3,
    'C': 2,
    'D': 1,
    'E': 0
}

graph = Graph(adjac_lis, heuristic)

# tìm đường từ A → E
graph.a_star_algorithm('A', 'E')


 1  2  3  4 
 5  6  7  8 
 9 10 11  0 
13 14 15 12 

 1  2  3  4 
 5  6  7  8 
 9 10 11 12 
13 14 15  0 

Đường đi: ['A', 'B', 'E']
Chi phí: 3


['A', 'B', 'E']